# Tazama Data Lakehouse — Catalog & Browser

Runs against the **live** Hudi warehouse at `WAREHOUSE_ROOT` (default `/opt/Tazama_Warehouse`).

**What this notebook does:**
1. Starts a minimal Spark session (read-only, no ETL)
2. Auto-discovers every Hudi table by scanning for `.hoodie/` marker directories
3. Prints a summary catalog: path · row count · record key · latest commit time
4. Lets you deep-dive any table: full schema, sample rows, commit history, partition map

Run cells top-to-bottom. Each section after §3 is independent — skip what you don't need.

In [25]:
# §1 — Spark session (read-only, Hudi + S3A/Ozone support)
import os, json, pathlib, textwrap
from pyspark.sql import SparkSession

spark_jars   = os.environ.get("SPARK_JARS", "")
jar_list     = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access   = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret   = os.environ.get("S3A_SECRET_KEY", "")

builder = (
    SparkSession.builder
    .appName("tazama-lakehouse-catalog")
    .master("local[2]")          # 2 cores — enough for reads
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions",
            "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory",
            os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
)

if s3a_endpoint:
    builder = (
        builder
        .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
        .config("spark.hadoop.fs.s3a.access.key", s3a_access)
        .config("spark.hadoop.fs.s3a.secret.key", s3a_secret)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl",
                "org.apache.hadoop.fs.s3a.S3AFileSystem")
    )

try:
    spark.stop()
except NameError:
    pass

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready.")

Spark 3.4.2 ready.


26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
26/05/01 15:02:12 WARN Utils: Service 'SparkUI' could not bind on port 4049. Attempting port 4050.


Exception in thread "serve-DataFrame" java.net.SocketTimeoutException: Accept timed out
	at java.base/java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.base/java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:474)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:565)
	at java.base/java.net.ServerSocket.accept(ServerSocket.java:533)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)


In [30]:
# §2 — Discover all Hudi tables under WAREHOUSE_ROOT
#
# A Hudi table is identified by a .hoodie/ sub-directory.
# We walk the tree and collect every directory that contains one.

WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
print(f"Scanning: {WAREHOUSE_ROOT}")

hudi_tables = []   # list of (layer, name, abs_path)

for root, dirs, files in os.walk(WAREHOUSE_ROOT):
    if ".hoodie" in dirs:
        rel   = os.path.relpath(root, WAREHOUSE_ROOT)   # e.g. gold/transactions
        parts = pathlib.PurePosixPath(rel.replace("\\", "/")).parts
        layer = parts[0] if len(parts) >= 2 else "(root)"
        name  = "/".join(parts[1:]) if len(parts) >= 2 else parts[0]
        hudi_tables.append((layer, name, root))
        dirs[:] = []   # don't recurse into Hudi table directories

hudi_tables.sort()
print(f"Found {len(hudi_tables)} Hudi table(s):\n")
for layer, name, path in hudi_tables:
    print(f"  [{layer:8s}]  {name}")
    print(f"             {path}")

Scanning: /opt/Tazama_Warehouse
Found 48 Hudi table(s):

  [AI      ]  cluster
             /opt/Tazama_Warehouse/AI/cluster
  [AI      ]  fraud_detection_feature_map
             /opt/Tazama_Warehouse/AI/fraud_detection_feature_map
  [bronze  ]  account
             /opt/Tazama_Warehouse/bronze/account
  [bronze  ]  account_holder
             /opt/Tazama_Warehouse/bronze/account_holder
  [bronze  ]  alerts
             /opt/Tazama_Warehouse/bronze/alerts
  [bronze  ]  cases
             /opt/Tazama_Warehouse/bronze/cases
  [bronze  ]  network_map
             /opt/Tazama_Warehouse/bronze/network_map
  [bronze  ]  pacs002
             /opt/Tazama_Warehouse/bronze/pacs002
  [bronze  ]  pacs008
             /opt/Tazama_Warehouse/bronze/pacs008
  [bronze  ]  rules
             /opt/Tazama_Warehouse/bronze/rules
  [bronze  ]  tasks
             /opt/Tazama_Warehouse/bronze/tasks
  [bronze  ]  transactions
             /opt/Tazama_Warehouse/bronze/transactions
  [bronze  ]  typologies
    

In [31]:
# §3 — Summary catalog: row count + latest commit per table
#
# Reads .hoodie/hoodie.properties (instant, no Spark) for table name / record key,
# then uses Spark only for the row count so startup cost is paid once.

import time

def read_hoodie_props(table_path):
    """Parse .hoodie/hoodie.properties without Spark."""
    props_file = os.path.join(table_path, ".hoodie", "hoodie.properties")
    props = {}
    if os.path.isfile(props_file):
        with open(props_file) as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    k, _, v = line.partition("=")
                    props[k.strip()] = v.strip()
    return props

def latest_commit(table_path):
    """Return the most recent completed commit timestamp string, or None."""
    timeline_dir = os.path.join(table_path, ".hoodie")
    commits = sorted(
        f for f in os.listdir(timeline_dir)
        if f.endswith(".commit") or f.endswith(".deltacommit")
    )
    if commits:
        ts = commits[-1].split(".")[0]   # yyyyMMddHHmmss
        return f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
    return "(no commits)"

rows_data = []
for layer, name, path in hudi_tables:
    props      = read_hoodie_props(path)
    tbl_name   = props.get("hoodie.table.name", "?")
    record_key = props.get("hoodie.datasource.write.recordkey.field", "?")
    tbl_type   = props.get("hoodie.table.type", "?")
    last_commit = latest_commit(path)

    t0  = time.time()
    cnt = spark.read.format("hudi").load(path).count()
    elapsed = time.time() - t0

    rows_data.append({
        "layer":      layer,
        "table":      name,
        "hudi_name":  tbl_name,
        "type":       tbl_type,
        "record_key": record_key,
        "rows":       cnt,
        "last_commit": last_commit,
        "path":       path,
    })
    print(f"  {layer}/{name:30s}  {cnt:>8,} rows   last commit: {last_commit}")

print("\nCatalog complete.")

  AI/cluster                          196,366 rows   last commit: 2026-04-15 01:35:53
  AI/fraud_detection_feature_map           15 rows   last commit: 2026-04-15 01:35:41
  bronze/account                          345,312 rows   last commit: 2026-04-30 14:40:12
  bronze/account_holder                   346,372 rows   last commit: 2026-04-30 14:33:34
  bronze/alerts                           128,378 rows   last commit: 2026-04-30 14:40:52
  bronze/cases                            128,381 rows   last commit: 2026-04-30 12:45:08
  bronze/network_map                           13 rows   last commit: 2026-04-30 13:36:12
  bronze/pacs002                          250,253 rows   last commit: 2026-04-30 14:34:20
  bronze/pacs008                          196,532 rows   last commit: 2026-04-30 14:30:21
  bronze/rules                                339 rows   last commit: 2026-04-30 13:35:58
  bronze/tasks                            128,378 rows   last commit: 2026-04-30 12:45:35
  bronze/transacti

26/05/01 15:07:08 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/silver/alerts_dlq


  silver/cases                            128,381 rows   last commit: 2026-04-30 12:45:13
  silver/cases_dlq                              0 rows   last commit: 2026-04-13 03:19:06


26/05/01 15:07:08 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/silver/cases_dlq


  silver/network_map                           13 rows   last commit: 2026-04-30 13:36:14
  silver/pacs002                          250,253 rows   last commit: 2026-04-30 14:34:42
  silver/pacs008                          196,532 rows   last commit: 2026-04-30 14:31:00
  silver/rules                                307 rows   last commit: 2026-04-30 13:36:00
  silver/tasks                            128,378 rows   last commit: 2026-04-30 12:45:41
  silver/tasks_dlq                        128,409 rows   last commit: 2026-04-30 12:45:45
  silver/transactions                     754,295 rows   last commit: 2026-04-30 14:36:36
  silver/typologies                            42 rows   last commit: 2026-04-30 13:36:06


26/05/01 15:07:10 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/views/alert_history


  views/alert_history                          0 rows   last commit: 2026-04-30 15:15:26
  views/alert_navigator/header           128,465 rows   last commit: 2026-04-30 14:41:51
  views/alert_navigator/rules_triggered   387,974 rows   last commit: 2026-04-30 14:42:16
  views/alert_navigator/typologies_triggered   128,464 rows   last commit: 2026-04-30 14:42:05
  views/vw_counterparty_account_links   1,385,964 rows   last commit: 2026-04-30 15:10:19
  views/vw_transaction_detail            754,452 rows   last commit: 2026-04-30 14:42:34


  views/vw_transaction_history          8,893,343 rows   last commit: 2026-04-30 14:45:05
  views/vw_tx_network_accounts_edges     753,064 rows   last commit: 2026-04-30 15:05:21
  views/vw_tx_network_counterparties_edges   679,260 rows   last commit: 2026-04-30 15:07:58

Catalog complete.


In [ ]:
# §3b — Render catalog as a pandas DataFrame for a cleaner view
import pandas as pd

catalog_df = pd.DataFrame(rows_data)[["layer", "table", "type", "record_key", "rows", "last_commit"]]
catalog_df.sort_values(["layer", "table"]).reset_index(drop=True)

---
## Deep-dive a specific table

Set `TARGET` to any `layer/name` value from the catalog above, then run §4 onwards.

In [52]:
# §4 — Pick a table to inspect
TARGET = "gold/pacs002"    # ← change me

table_meta = next((r for r in rows_data if f"{r['layer']}/{r['table']}" == TARGET), None)
if table_meta is None:
    raise ValueError(f"{TARGET!r} not found in catalog. Available:\n"
                     + "\n".join(f"  {r['layer']}/{r['table']}" for r in rows_data))

df = spark.read.format("hudi").load(table_meta["path"])
print(f"Loaded  : {TARGET}")
print(f"Path    : {table_meta['path']}")
print(f"Rows    : {table_meta['rows']:,}")
print(f"Columns : {len(df.columns)}")

Loaded  : gold/pacs002
Path    : /opt/Tazama_Warehouse/gold/pacs002
Rows    : 250,253
Columns : 45


In [53]:
# §5 — Schema (hide Hudi internal columns by default)
SHOW_HUDI_INTERNALS = False   # set True to include _hoodie_* columns

if SHOW_HUDI_INTERNALS:
    df.printSchema()
else:
    from pyspark.sql.types import StructType
    visible = StructType([f for f in df.schema.fields
                          if not f.name.startswith("_hoodie_")])
    # Pretty-print manually so it looks the same as printSchema()
    def _print_schema(schema, indent=0):
        for field in schema.fields:
            nullable = "nullable" if field.nullable else "not nullable"
            dtype = field.dataType
            if hasattr(dtype, "fields"):   # StructType
                print(" " * indent + f"|-- {field.name}: struct ({nullable})")
                _print_schema(dtype, indent + 4)
            elif hasattr(dtype, "elementType"):   # ArrayType
                print(" " * indent + f"|-- {field.name}: array ({nullable})")
            else:
                print(" " * indent + f"|-- {field.name}: {dtype.simpleString()} ({nullable})")
    print(f"root ({len(visible.fields)} business columns, _hoodie_* hidden)")
    _print_schema(visible)

root (40 business columns, _hoodie_* hidden)
|-- pk: string (nullable)
|-- tenant_id: string (nullable)
|-- message_id: string (nullable)
|-- end_to_end_id: string (nullable)
|-- credttm_raw: string (nullable)
|-- credttm_ts: timestamp (nullable)
|-- tx_type: string (not nullable)
|-- tx_msg_id: string (nullable)
|-- tx_status: string (nullable)
|-- tx_amount: double (nullable)
|-- tx_ccy: string (nullable)
|-- instg_mmb_id: string (nullable)
|-- instd_mmb_id: string (nullable)
|-- charge_count: int (not nullable)
|-- event_ts: timestamp (nullable)
|-- tx_event_ts: timestamp (nullable)
|-- event_date: date (nullable)
|-- event_to_ingest_ms: bigint (nullable)
|-- tx_tenant_id: string (nullable)
|-- dc_cdtr_id: string (nullable)
|-- dc_dbtr_id: string (nullable)
|-- dc_cre_dt_tm: timestamp (nullable)
|-- dc_instd_amt: double (nullable)
|-- dc_instd_ccy: string (nullable)
|-- dc_xchg_rate: string (nullable)
|-- dc_cdtr_acct_id: string (nullable)
|-- dc_dbtr_acct_id: string (nullable)
|-- 

In [63]:
# §6 — Sample rows (most recent 10 by event time)
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DateType

def to_pandas_safe(sdf):
    """Cast timestamp/date cols to string to avoid pandas 2.x tz conversion error."""
    for field in sdf.schema.fields:
        if isinstance(field.dataType, (TimestampType, DateType)):
            sdf = sdf.withColumn(field.name, F.col(field.name).cast("string"))
    return sdf.toPandas()

# Prefer a business event timestamp over the Hudi internal commit time.
# This ensures the most recently TMS-processed records (with DataCache populated)
# appear first rather than the most recently ingested Hudi commits.
_EVENT_TS_CANDIDATES = ["credttm", "cre_dt_tm", "dc_cre_dt_tm", "event_ts", "created_at", "tx_dt"]
_sort_col = next(
    (c for c in _EVENT_TS_CANDIDATES if c in df.columns),
    "_hoodie_commit_time"   # fallback: Hudi internal ingestion timestamp
)

business_cols = [c for c in df.columns if not c.startswith("_hoodie_")]
sample = (
    df.select("_hoodie_commit_time", *business_cols)
      .orderBy(_sort_col, ascending=False)
      .limit(10)
)
print(f"Sorted by: {_sort_col} (descending)")
to_pandas_safe(sample)


Sorted by: dc_cre_dt_tm (descending)


,_hoodie_commit_time,pk,tenant_id,message_id,end_to_end_id,credttm_raw,credttm_ts,tx_type,tx_msg_id,tx_status,...,grp_cre_dt_tm,accptnc_dt_tm,orgnl_instr_id,orgnl_end_to_end_id,status_reason_code,charge_total_amount,charge_currency_count,charge_currency_hint,record_hash,ingested_at_ts
0,20260430143512381,fa10268fe6c363ed93414454fb10b64836de6c44414992...,TAZAMA,30e84534ee3a4e53bc49bf85bd9c1504,7586b82b775b4984b149b41c81ba1d11,2026-04-30T14:30:10.586Z,2026-04-30 14:30:10.586,pacs.002.001.12,30e84534ee3a4e53bc49bf85bd9c1504,ACCC,...,2026-04-30 14:30:10.586,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,7586b82b775b4984b149b41c81ba1d11,None,0.0,1,USD,776ee699ef79f745e4283599fc1ab99d4469cb558ffc39...,2026-04-30 14:34:20.753119
1,20260430143512381,616336deacce04fa37ff0a8105bed0899d5f9c52df2291...,TAZAMA,23142fe3929144438154a685a87a9a24,2cf6ade26a0b4dcf8a34e1e77379e4e5,2026-04-30T14:30:03.754Z,2026-04-30 14:30:03.754,pacs.002.001.12,23142fe3929144438154a685a87a9a24,ACCC,...,2026-04-30 14:30:03.754,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,2cf6ade26a0b4dcf8a34e1e77379e4e5,None,0.0,1,USD,0a281917fa5ab082fa4b3aa8ff6c156de8729e7bf6902c...,2026-04-30 14:34:20.753119
2,20260430143512381,fd55463faed7f4ce01811c7caf4506bdcd15c02cbdf721...,TAZAMA,36a8e4d710024a51991539f0324c5387,c8e596f0c4b541098b3efb19147b6b8e,2026-04-30T14:29:22.070Z,2026-04-30 14:29:22.07,pacs.002.001.12,36a8e4d710024a51991539f0324c5387,ACCC,...,2026-04-30 14:29:22.07,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,c8e596f0c4b541098b3efb19147b6b8e,None,0.0,1,USD,53e2d6c319b6b887eb3bfd37710286f8d1d211e81c9a08...,2026-04-30 14:34:20.753119
3,20260430143512381,eedf2d5002d9bfa68b820e1a0ee1d720901d91b05cbd3b...,TAZAMA,71750cc3cd33446a83bd66d97da07e13,f749f6f53faa4539b5bfe911eebcf1d8,2026-04-30T09:50:43.513Z,2026-04-30 09:50:43.513,pacs.002.001.12,71750cc3cd33446a83bd66d97da07e13,ACCC,...,2026-04-30 09:50:43.513,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,f749f6f53faa4539b5bfe911eebcf1d8,None,0.0,1,USD,1976f3c94575009f771837bd0c95f4bc1d1dcb6ffb511f...,2026-04-30 09:57:17.055698
4,20260430143512381,b7fbd43700e1be090f43079805c9ac6c9b54fbc58ad272...,TAZAMA,c15429fee8204fe698042cf8ba44b55c,478e65ec70004f10ae239cea39eda5fb,2026-04-29T15:39:55.034Z,2026-04-29 15:39:55.034,pacs.002.001.12,c15429fee8204fe698042cf8ba44b55c,ACCC,...,2026-04-29 15:39:55.034,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,478e65ec70004f10ae239cea39eda5fb,None,0.0,1,USD,647a7f24e4d7e39cf665b5e1a6f84760c25aceb345161a...,2026-04-29 16:31:46.329697
5,20260430143512381,a4b58670a858a6975bb70721f4e55146958bae0e1fe143...,TAZAMA,cc593852567142a684351e1da216687f,0403e75f71c7472d9414720ae9f785dd,2026-04-29T15:38:29.270Z,2026-04-29 15:38:29.27,pacs.002.001.12,cc593852567142a684351e1da216687f,ACCC,...,2026-04-29 15:38:29.27,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,0403e75f71c7472d9414720ae9f785dd,None,0.0,1,USD,b9a2deddcc3477c11abfcc4f4ceef96f67aa7f9b831dd3...,2026-04-29 16:19:49.905011
6,20260430143512381,2b82aa5192c429b4df8269479c75980af887500f79a36d...,TAZAMA,2d16241fe6a34c499c198df32b294285,49984532e8bd417e9a9f840f3fd41a48,2026-04-29T15:22:00.646Z,2026-04-29 15:22:00.646,pacs.002.001.12,2d16241fe6a34c499c198df32b294285,ACCC,...,2026-04-29 15:22:00.646,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,49984532e8bd417e9a9f840f3fd41a48,None,0.0,1,USD,cfea12810901677178e3362c977250ac4512f759835d39...,2026-04-29 16:09:29.207991
7,20260430143512381,b5ae9c101f30c90c3b0f81cd48dccf7c04152cc63c4188...,TAZAMA,137462a194be47c1a5ef6e9b7ad6cf86,78882ee4492d441dbf1c4a7e304b84ef,2026-04-29T15:19:52.105Z,2026-04-29 15:19:52.105,pacs.002.001.12,137462a194be47c1a5ef6e9b7ad6cf86,ACCC,...,2026-04-29 15:19:52.105,2023-06-02 07:52:31,5ab4fc7355de4ef8a75b78b00a681ed2,78882ee4492d441dbf1c4a7e304b84ef,None,0.0,1,USD,2cb04c4dc9e34e4ea5f620101e2c33b4939dc4f7541749...,2026-04-29 15:58:25.750024
8,20260430143512381,4c2f9453d4e149595c1622fe31101bb31b64219833b232...,TAZAMA,37e568a031254835aab0459e0

In [ ]:
# §6b — Single record detail (all fields, fully expanded)
#
# Set MESSAGE_ID and TENANT_ID to pin a specific record.
# Leave both empty to show the most-recent record (by the same sort column as §6).
#
# The DataFrame is transposed so each field is on its own row — making
# long values (JSON payloads, nested structs, DataCache blocks) easy to read.

MESSAGE_ID = ""   # e.g. "0db3c737447347b0bf38dc466e0743c6"
TENANT_ID  = ""   # e.g. "PAYSYSLABS"

# Determine the record key field name from Hudi properties (fallback: common names)
_props = read_hoodie_props(table_meta["path"])
_rk    = _props.get("hoodie.datasource.write.recordkey.field", "")
_ID_CANDIDATES = [c for c in ([_rk] + ["msg_id", "message_id", "transaction_id", "id"])
                  if c in df.columns]
_id_col = _ID_CANDIDATES[0] if _ID_CANDIDATES else None

if MESSAGE_ID and _id_col:
    filtered = df.filter(F.col(_id_col) == MESSAGE_ID)
    if TENANT_ID:
        _TENANT_CANDIDATES = [c for c in ["tenant_id", "tx_tenant_id", "tenantid"] if c in df.columns]
        if _TENANT_CANDIDATES:
            filtered = filtered.filter(F.col(_TENANT_CANDIDATES[0]) == TENANT_ID)
    row_df = filtered.limit(1)
    print(f"Filtered by: {_id_col} = {MESSAGE_ID!r}"
          + (f"  +  tenant = {TENANT_ID!r}" if TENANT_ID else ""))
else:
    # Fall back to most-recent record using the same sort column as §6
    row_df = df.orderBy(_sort_col, ascending=False).limit(1)
    print(f"Showing most-recent record (sorted by: {_sort_col})")
    print("  → Set MESSAGE_ID above to pin a specific record.")

single_row = to_pandas_safe(row_df).T.rename(columns={0: "value"})

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(single_row)


,value
_hoodie_commit_time,20260430143512381
_hoodie_commit_seqno,20260430143512381_0_0
_hoodie_record_key,8f299f6b264b27aacb1dfa6579b44824e7a876620e7a9406f9bfb6147832cf12
_hoodie_partition_path,
_hoodie_file_name,c5443921-4ff0-4a52-9624-311f809c1e5a-0_0-13120-127928_20260430143512381.parquet
pk,8f299f6b264b27aacb1dfa6579b44824e7a876620e7a9406f9bfb6147832cf12
tenant_id,DEFAULT
message_id,M3754482414
end_to_end_id,94c0d591-cff3-4333-b333-449e02d96ae
credttm_raw,2026-01-28T19:08:02.589+03:00


In [ ]:
# §7 — Commit history (how many rows landed in each commit)
(
    df.groupBy(F.substring("_hoodie_commit_time", 1, 12).alias("commit_minute"))
      .count()
      .orderBy("commit_minute", ascending=False)
      .limit(30)
      .toPandas()
)

In [ ]:
# §8 — Column-level statistics (nulls, distinct values, numeric ranges)
#
# Runs spark.sql describe() on non-struct, non-array columns.

from pyspark.sql.types import StructType, ArrayType, MapType

scalar_cols = [
    f.name for f in df.schema.fields
    if not f.name.startswith("_hoodie_")
    and not isinstance(f.dataType, (StructType, ArrayType, MapType))
]

stats = df.select(*scalar_cols).describe()
to_pandas_safe(stats)

In [58]:
# §9 — Partition map
#
# Reads _hoodie_partition_path (written on every Hudi row) instead of
# hoodie.properties, so the result is accurate even when NonpartitionedKeyGenerator
# was used — including cases where a partition field was configured but the
# generator class overrides it at write time.

from IPython.display import display

partition_counts = (
    df.groupBy("_hoodie_partition_path")
      .count()
      .orderBy("count", ascending=False)
      .limit(50)
      .toPandas()
)

unique_paths = set(partition_counts["_hoodie_partition_path"].tolist())

if unique_paths <= {"", "default"}:
    # Single empty/placeholder path — table is not partitioned
    props = read_hoodie_props(table_meta["path"])
    kg    = props.get("hoodie.datasource.write.keygenerator.class", "").split(".")[-1]
    pf    = props.get("hoodie.datasource.write.partitionpath.field", "(none)")
    print("Table is not partitioned.")
    print(f"  KeyGenerator             : {kg or '(not set)'}")
    print(f"  Partition field in props : {pf}")
    print(f"  _hoodie_partition_path   : {repr(next(iter(unique_paths), ''))}")
else:
    props = read_hoodie_props(table_meta["path"])
    pf    = props.get("hoodie.datasource.write.partitionpath.field", "(not set in properties)")
    print(f"Partition field (from properties) : {pf}")
    print(f"Distinct partition paths          : {len(unique_paths)}")
    display(partition_counts)


Table is not partitioned.
  KeyGenerator             : (not set)
  Partition field in props : (none)
  _hoodie_partition_path   : ''


In [59]:
# §10 — Register all tables as Spark SQL temp views
#
# After this cell you can run arbitrary SQL in the next cell.
# View names: <layer>_<table>  (slashes → underscores)

registered = []
for r in rows_data:
    view_name = f"{r['layer']}_{r['table'].replace('/', '_')}"
    spark.read.format("hudi").load(r["path"]).createOrReplaceTempView(view_name)
    registered.append(view_name)

print("Registered temp views:")
for v in registered:
    print(f"  {v}")

26/05/01 16:06:30 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/silver/alerts_dlq
26/05/01 16:06:30 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/silver/cases_dlq
26/05/01 16:06:31 WARN TableSchemaResolver: Could not find any data file written for commit, so could not get schema for table file:/opt/Tazama_Warehouse/views/alert_history


Registered temp views:
  AI_cluster
  AI_fraud_detection_feature_map
  bronze_account
  bronze_account_holder
  bronze_alerts
  bronze_cases
  bronze_network_map
  bronze_pacs002
  bronze_pacs008
  bronze_rules
  bronze_tasks
  bronze_transactions
  bronze_typologies
  gold_account
  gold_account_holder
  gold_alerts
  gold_biar_metrics
  gold_cases
  gold_network_map
  gold_pacs002
  gold_pacs008
  gold_rules
  gold_tasks
  gold_transactions
  gold_typologies
  silver_account
  silver_account_holder
  silver_alerts
  silver_alerts_dlq
  silver_cases
  silver_cases_dlq
  silver_network_map
  silver_pacs002
  silver_pacs008
  silver_rules
  silver_tasks
  silver_tasks_dlq
  silver_transactions
  silver_typologies
  views_alert_history
  views_alert_navigator_header
  views_alert_navigator_rules_triggered
  views_alert_navigator_typologies_triggered
  views_vw_counterparty_account_links
  views_vw_transaction_detail
  views_vw_transaction_history
  views_vw_tx_network_accounts_edges
  vi

In [60]:
# §11 — Spark SQL playground
#
# Change the query below. Use the view names printed in §10.

SQL = """
SELECT
    SUBSTRING(_hoodie_commit_time, 1, 8) AS commit_day,
    COUNT(*)                             AS row_count
FROM gold_transactions
GROUP BY 1
ORDER BY 1 DESC
LIMIT 30
"""

to_pandas_safe(spark.sql(SQL))

,commit_day,row_count
0,20260430,754295


In [61]:
# §11b — Cross-table join: transactions → debtor/creditor accounts
#
# `dc_cdtr_acct_id` and `dc_dbtr_acct_id` live in `gold/pacs002`, NOT in
# `gold/transactions`.  They come from the Tazama TMS `DataCache` enrichment
# block embedded in the stored pacs.002 message — not from standard ISO 20022
# fields.  The full lookup chain is:
#
#   gold_transactions  ─(end_to_end_id + tenant_id)─▶  gold_pacs002
#   gold_pacs002       ─(dc_cdtr_acct_id)────────────▶  gold_account   (creditor account detail)
#   gold_pacs002       ─(dc_dbtr_acct_id)────────────▶  gold_account   (debtor account detail)
#   gold_pacs002       ─(dc_cdtr_id)──────────────────▶ gold_account_holder.counterparty_id
#   gold_pacs002       ─(dc_dbtr_id)──────────────────▶ gold_account_holder.counterparty_id
#
# Requires §10 (temp views) to have been run first.

SQL_ACCT_LOOKUP = """
SELECT
    t.transaction_id,
    t.end_to_end_id,
    t.tenant_id,
    t.tx_type,
    t.tx_amount,
    t.tx_ccy,
    t.tx_status,
    t.event_ts,

    -- Creditor (payee) account fields from DataCache
    p.dc_cdtr_acct_id                   AS cdtr_account_id,
    p.dc_cdtr_id                        AS cdtr_party_id,
    cdtr_acct.account_id                AS cdtr_acct_confirmed,

    -- Debtor (payer) account fields from DataCache
    p.dc_dbtr_acct_id                   AS dbtr_account_id,
    p.dc_dbtr_id                        AS dbtr_party_id,
    dbtr_acct.account_id                AS dbtr_acct_confirmed,

    -- DataCache amount (pre-resolved by TMS; may differ from pacs.008 InstructedAmount)
    p.dc_instd_amt                      AS dc_amount,
    p.dc_instd_ccy                      AS dc_ccy

FROM       gold_transactions            AS t

-- Link to pacs002 via end-to-end ID + tenant (DataCache fields only exist on gold_pacs002)
INNER JOIN gold_pacs002                 AS p
        ON  p.orgnl_end_to_end_id = t.end_to_end_id
        AND p.tx_tenant_id        = t.tenant_id

-- Resolve creditor account from gold/account
LEFT  JOIN gold_account                 AS cdtr_acct
        ON  cdtr_acct.account_id = p.dc_cdtr_acct_id
        AND cdtr_acct.tenant_id  = t.tenant_id

-- Resolve debtor account from gold/account
LEFT  JOIN gold_account                 AS dbtr_acct
        ON  dbtr_acct.account_id = p.dc_dbtr_acct_id
        AND dbtr_acct.tenant_id  = t.tenant_id

ORDER BY t.event_ts DESC
LIMIT 20
"""

to_pandas_safe(spark.sql(SQL_ACCT_LOOKUP))


,transaction_id,end_to_end_id,tenant_id,tx_type,tx_amount,tx_ccy,tx_status,event_ts,cdtr_account_id,cdtr_party_id,cdtr_acct_confirmed,dbtr_account_id,dbtr_party_id,dbtr_acct_confirmed,dc_amount,dc_ccy
0,540665922,7586b82b775b4984b149b41c81ba1d11,TAZAMA,pacs.002.001.12,NaN,None,ACCC,2026-04-30 14:30:10.586,cdtrAcct_e37ed3777d4448309378708e952249b5MSISD...,cdtr_e6a31ffab91f4dbe9df3d66bb81a16faTAZAMA_EID,cdtrAcct_e37ed3777d4448309378708e952249b5MSISD...,dbtrAcct_b159394925cc478f8193b7413e18d666MSISD...,dbtr_9fe0981b6e144196953dbcf90825a415TAZAMA_EID,dbtrAcct_b159394925cc478f8193b7413e18d666MSISD...,314.19,XTS
1,187136095,2cf6ade26a0b4dcf8a34e1e77379e4e5,TAZAMA,pacs.002.001.12,NaN,None,ACCC,2026-04-30 14:30:03.754,cdtrAcct_f5649f5c37f14614941e85432b9e6d94MSISD...,cdtr_ac4f299213d545ffb0df14a8c8fd5046TAZAMA_EID,cdtrAcct_f5649f5c37f14614941e85432b9e6d94MSISD...,dbtrAcct_fcbed38acea047afb373b43fffd3090dMSISD...,dbtr_0ffa63cd53d5438d9ee5435ca148b5b7TAZAMA_EID,dbtrAcct_fcbed38acea047afb373b43fffd3090dMSISD...,418.58,XTS
2,759580340,c8e596f0c4b541098b3efb19147b6b8e,TAZAMA,pacs.002.001.12,NaN,None,ACCC,2026-04-30 14:29:22.07,cdtrAcct_61bb66b485644c03920d365aae7528a0MSISD...,cdtr_470af92afc3b47b791522adc80009793TAZAMA_EID,cdtrAcct_61bb66b485644c03920d365aae7528a0MSISD...,dbtrAcct_8d493113dfb04d368b8f47232fd3a15eMSISD...,dbtr_a3c7512ac6c644b6be243f7f68fa8c25TAZAMA_EID,dbtrAcct_8d493113dfb04d368b8f47232fd3a15eMSISD...,55.66,XTS
3,877820395,7586b82b775b4984b149b41c81ba1d11,TAZAMA,pacs.008.001.10,314.19,XTS,None,2026-04-30 14:25:10.586,cdtrAcct_e37ed3777d4448309378708e952249b5MSISD...,cdtr_e6a31ffab91f4dbe9df3d66bb81a16faTAZAMA_EID,cdtrAcct_e37ed3777d4448309378708e952249b5MSISD...,dbtrAcct_b159394925cc478f8193b7413e18d666MSISD...,dbtr_9fe0981b6e144196953dbcf90825a415TAZAMA_EID,dbtrAcct_b159394925cc478f8193b7413e18d666MSISD...,314.19,XTS
4,303402145,2cf6ade26a0b4dcf8a34e1e77379e4e5,TAZAMA,pacs.008.001.10,418.58,XTS,None,2026-04-30 14:25:03.754,cdtrAcct_f5649f5c37f14614941e85432b9e6d94MSISD...,cdtr_ac4f299213d545ffb0df14a8c8fd5046TAZAMA_EID,cdtrAcct_f5649f5c37f14614941e85432b9e6d94MSISD...,dbtrAcct_fcbed38acea047afb373b43fffd3090dMSISD...,dbtr_0ffa63cd53d5438d9ee5435ca148b5b7TAZAMA_EID,dbtrAcct_fcbed38acea047afb373b43fffd3090dMSISD...,418.58,XTS
5,364608762,c8e596f0c4b541098b3efb19147b6b8e,TAZAMA,pacs.008.001.10,55.66,XTS,None,2026-04-30 14:24:22.07,cdtrAcct_61bb66b485644c03920d365aae7528a0MSISD...,cdtr_470af92afc3b47b791522adc80009793TAZAMA_EID,cdtrAcct_61bb66b485644c03920d365aae7528a0MSISD...,dbtrAcct_8d493113dfb04d368b8f47232fd3a15eMSISD...,dbtr_a3c7512ac6c644b6be243f7f68fa8c25TAZAMA_EID,dbtrAcct_8d493113dfb04d368b8f47232fd3a15eMSISD...,55.66,XTS
6,402844612,f749f6f53faa4539b5bfe911eebcf1d8,TAZAMA,pacs.002.001.12,NaN,None,ACCC,2026-04-30 09:50:43.513,cdtrAcct_674e3ef52f1946d3bcd2e90abe74f5acMSISD...,cdtr_08b2a26a59bc419693a86d9bee6f25b8TAZAMA_EID,cdtrAcct_674e3ef52f1946d3bcd2e90abe74f5acMSISD...,dbtrAcct_9a87484929d34fd3b5b54d958f05f3e6MSISD...,dbtr_d08732a88977468d97e69bce431b5c1dTAZAMA_EID,dbtrAcct_9a87484929d34fd3b5b54d958f05f3e6MSISD...,877.63,XTS
7,201268492,f749f6f53faa4539b5bfe911eebcf1d8,TAZAMA,pacs.008.001.10,877.63,XTS,None,2026-04-30 09:45:43.513,cdtrAcct_674e3ef52f1946d3bcd2e90abe74f5acMSISD...,cdtr_08b2a26a59bc419693a86d9bee6f25b8TAZAMA_EID,cdtrAcct_674e3ef52f1946d3bcd2e90abe74f5acMSISD...,dbtrAcct_9a87484929d34fd3b5b54d958f05f3e6MSISD...,dbtr_d08732a88977468d97e69bce431b5c1dTAZAMA_EID,dbtrAcct_9a87484929d34fd3b5b54d958f05f3e6MSISD...,877.63,XTS
8,209060508,478e65ec70004f10ae239cea39eda5fb,TAZAMA,pacs.002.001.12,NaN,None,ACCC,2026-04-29 15:39:55.034,cdtrAcct_14415f93417e4b94b006f20fb93a5123MSISD...,cdtr_b6ffd7b0318c4dfd9dc4f7bb9b06e6aeTAZAMA_EID,cdtrAcct_14415f93417e4b94b006f20fb93a5123MSISD...,dbtrAcct_2bba63a7f5e34e1fb8a96224c919e10bMSISD...,dbtr_2d7d7ca3b56d46099d1ef91e6de11dfaTAZAMA_EID,dbtrAcct_2bba63a7f5e34e1fb8a96224c919e10bMSISD...,272.08,XTS
9,340280377,0403e75f71c7472d9414720ae9f785dd,TAZAMA,pacs.002.001.12,NaN,None,AC

In [ ]:
# §12 — Hudi timeline (raw .commit files) for the target table
#
# Shows every action recorded in the Hudi timeline directory.
#
# Hudi 0.14+ uses 17-digit timestamps (yyyyMMddHHmmssSSS).
# Earlier versions used 14-digit timestamps (yyyyMMddHHmmss).
# The regex matches both.

import re
from IPython.display import display

timeline_dir = os.path.join(table_meta["path"], ".hoodie")
timeline_files = sorted(
    f for f in os.listdir(timeline_dir)
    if re.match(r"^\d{14,}\.", f)
)

if not timeline_files:
    print(f"No timeline files found in {timeline_dir}")
    print("Directory contents:")
    for entry in sorted(os.listdir(timeline_dir)):
        print(f"  {entry}")
else:
    rows_tl = []
    for f in timeline_files:
        ts_raw, _, rest = f.partition(".")
        # Normalise action label: strip trailing qualifiers into readable tags
        action = (
            rest
            .replace(".requested", " (requested)")
            .replace(".inflight",  " (inflight)")
        )
        ts = ts_raw
        ts_fmt = (
            f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
            + (f".{ts[14:]}" if len(ts) > 14 else "")
            if len(ts) >= 14 else ts
        )
        size_kb = os.path.getsize(os.path.join(timeline_dir, f)) / 1024
        rows_tl.append({
            "timestamp": ts_fmt,
            "action":    action,
            "file_KB":   round(size_kb, 1),
        })

    display(pd.DataFrame(rows_tl).tail(30))


In [ ]:
# §13 — Time-travel: read the table as of a specific commit
#
# Set AS_OF_COMMIT to a timestamp from the range shown when you leave it empty.
# Hudi 0.14+ uses 17-digit timestamps (yyyyMMddHHmmssSSS).
# A 14-digit prefix is also accepted and will match the first qualifying instant.
# Leave empty to see the valid range without reading any data.

AS_OF_COMMIT = ""   # e.g. "20250101120000123"

# ── Resolve completed commits from the .hoodie timeline ──────────────────────
timeline_dir = os.path.join(table_meta["path"], ".hoodie")
completed = sorted(
    f.split(".")[0]
    for f in os.listdir(timeline_dir)
    if re.match(r"^\d{14,}\.commit$", f)   # only fully completed commits
)

def fmt_ts(ts):
    """Pretty-print a raw Hudi timestamp string."""
    return (
        f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
        + (f".{ts[14:]}" if len(ts) > 14 else "")
        if len(ts) >= 14 else ts
    )

if not completed:
    print("No completed commits found in the timeline — the table may still be empty.")

elif not AS_OF_COMMIT:
    # ── Guidance mode: show bounds and a sample of available instants ────────
    print("AS_OF_COMMIT is not set.  Set it to any value in the range below.\n")
    print(f"  Earliest commit : {completed[0]}")
    print(f"                    ({fmt_ts(completed[0])})")
    print(f"  Latest commit   : {completed[-1]}")
    print(f"                    ({fmt_ts(completed[-1])})")
    print(f"\n  Total completed commits : {len(completed)}")
    print("\n  Last 5 commits (most useful for time-travel):")
    for c in completed[-5:]:
        print(f"    {c}  ({fmt_ts(c)})")

else:
    # ── Execution mode: validate then read ───────────────────────────────────
    matched = [c for c in completed if c.startswith(AS_OF_COMMIT)]

    if not matched:
        print(f"ERROR: No completed commit found matching {AS_OF_COMMIT!r}\n")
        print(f"  Valid range:  {completed[0]}  →  {completed[-1]}")
        print(f"  ({fmt_ts(completed[0])}  →  {fmt_ts(completed[-1])})")
        print(f"\n  Total completed commits : {len(completed)}")
        print("\n  Last 5 commits:")
        for c in completed[-5:]:
            print(f"    {c}  ({fmt_ts(c)})")
    else:
        instant = matched[0]
        print(f"Reading snapshot at : {instant}  ({fmt_ts(instant)})")
        print(f"Valid range         : {completed[0]}  →  {completed[-1]}\n")
        df_snapshot = (
            spark.read
            .format("hudi")
            .option("as.of.instant", instant)
            .load(table_meta["path"])
        )
        print(f"Rows at {instant}: {df_snapshot.count():,}")
        display(to_pandas_safe(df_snapshot.limit(5)))


In [ ]:
# §14 — Tear down Spark session when done
spark.stop()
print("Spark stopped.")